<a href="https://colab.research.google.com/github/Tejeshwini89/Gen-AI/blob/main/Summarization_Function.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install required libraries (uncomment and run if not already installed)
!pip install transformers torch sentencepiece

# 2. Import libraries
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
import textwrap

# 3. Check for GPU availability
device = 0 if torch.cuda.is_available() else -1
print(f"Using device: {'GPU' if device == 0 else 'CPU' }")

# 4. Model selection (choose one)
# Available models for summarization:
# - facebook/bart-large-cnn (BART, good for news articles)
# - t5-small, t5-base, t5-large (T5, versatile)
# - google/pegasus-cnn_dailymail (PEGASUS, excellent for news)
# - google/pegasus-xsum (PEGASUS, for very short summaries)
MODEL_NAME = "facebook/bart-large-cnn" # Change this to try other models
# 5. Load tokenizer and model
print(f"Loading model: {MODEL_NAME}")
tokenizer = AutoTokenizer. from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM. from_pretrained(MODEL_NAME)



Using device: GPU
Loading model: facebook/bart-large-cnn


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

In [ ]:
# Move model to GPU if available
if device == 0:
    model = model.to("cuda") #Compute Unified Distribute Architecture

# 6. Summarization function
def summarize(text, max_length=150, min_length=40, do_sample=False):

    # Generate a summary of the input text using the loaded model.
    # Handles long texts by splitting into chunks if necessary.

    # Tokenize input
    inputs = tokenizer.encode(text, return_tensors="pt", truncation=True, max_length=1024)
    if device == 0:
        inputs = inputs.to("cuda") # Move inputs to GPU if available

    # Generate summary
    summary_ids = model.generate(
        inputs,
        max_length=max_length,
        min_length=min_length,
        length_penalty=2.0,
        num_beams=4,
        do_sample=do_sample,
        early_stopping=True
    )
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

In [ ]:
# 7. Alternative: Use the Hugging Face pipeline (simpler)
# Uncomment the following lines if you prefer the pipeline approach.
# summarizer = pipeline("summarization", model=MODEL_NAME, device=device)
# def summarize_pipeline(text, max_length=150, min_length=40):
#    return summarizer(text, max_length=max_length, min_length=min_length) [0]['summary_text']

# 8. Example text (you can replace this with your own)
EXAMPLE_TEXT = """
The United States and Iran reached an 11th-hour cease-fire deal on Tuesday evening, hours after President
Trump threatened to start wiping out Iran’s “whole civilization” if it did not allow commercial shipping to
pass safely through the Strait of Hormuz."""

print("Original Text:\n", textwrap.fill(EXAMPLE_TEXT, width=80))
print("\n---\n")

Original Text:
  The United States and Iran reached an 11th-hour cease-fire deal on Tuesday
evening, hours after President Trump threatened to start wiping out Iran’s
“whole civilization” if it did not allow commercial shipping to pass safely
through the Strait of Hormuz.

---

